# RetailOps 360 — 02_Revenue & Payment Reconciliation Analysis

GitHub-readable copy reconstructed from the executed Databricks HTML export. Saved tabular output previews are limited to 100 rows each for a readable GitHub display. 

# Segment 2 - Revenue & Payment Reconciliation Analysis

## Analysis 04 — Multiple Payment Transaction Analysis
### Business question: 
### Which orders have multiple payment transactions, and what is the combined payment amount for each order?

In [ ]:
%sql
SELECT
    o.order_id,
    o.order_date,
    o.order_amount,
    COUNT(p.payment_id) AS payment_count,
    ROUND(SUM(p.payment_amount), 2) AS total_payment_amount
FROM retailops360.source.orders AS o
INNER JOIN retailops360.source.payments AS p
    ON o.order_id = p.order_id
GROUP BY
    o.order_id,
    o.order_date,
    o.order_amount
HAVING COUNT(p.payment_id) > 1
ORDER BY payment_count DESC, o.order_id;

order_id,order_date,order_amount,payment_count,total_payment_amount
3,2024-08-26,792.00,2,792.00
8,2024-05-13,1044.00,2,1044.00
13,2025-08-05,1751.00,2,1751.00
14,2026-01-02,2507.00,2,2507.00
25,2024-04-26,1188.00,2,1188.00
35,2026-09-06,1667.00,2,1667.00
37,2025-04-25,3213.00,2,3213.00
42,2026-05-16,2987.00,2,2987.00
46,2024-10-25,3447.00,2,3447.00
52,2025-12-03,3588.00,2,3588.00


## Analysis 05 — Order-to-Payment Reconciliation
### Business question: 
### Which orders are unpaid, underpaid, fully paid, or overpaid when total captured payments are compared with the order amount?

In [ ]:
%sql
WITH payment_summary AS (
SELECT
        order_id,
        COUNT(payment_id) AS payment_count,
        SUM(payment_amount) AS total_paid
FROM retailops360.source.payments
    WHERE payment_status = 'Captured'
    GROUP BY order_id
),

order_reconciliation AS (
    SELECT
        o.order_id,
        o.order_date,
        o.order_amount,
        COALESCE(p.payment_count, 0) AS payment_count,
        COALESCE(p.total_paid, 0) AS total_paid
    FROM retailops360.source.orders AS o
    LEFT JOIN payment_summary AS p
        ON o.order_id = p.order_id
)

SELECT
    order_id,
    order_date,
    order_amount,
    payment_count,
    ROUND(total_paid, 2) AS total_paid,
    ROUND(order_amount - total_paid, 2) AS balance_difference,
    CASE
        WHEN total_paid = 0 THEN 'Unpaid'
        WHEN total_paid < order_amount THEN 'Underpaid'
        WHEN total_paid = order_amount THEN 'Fully Paid'
        ELSE 'Overpaid'
    END AS reconciliation_status
FROM order_reconciliation
ORDER BY order_id;

order_id,order_date,order_amount,payment_count,total_paid,balance_difference,reconciliation_status
1,2025-04-26,2832.00,1,2832.00,0.00,Fully Paid
2,2025-01-02,1078.00,1,1078.00,0.00,Fully Paid
3,2024-08-26,792.00,2,792.00,0.00,Fully Paid
4,2026-06-18,1614.00,1,1614.00,0.00,Fully Paid
5,2024-11-29,821.00,1,821.00,0.00,Fully Paid
6,2024-09-18,1953.00,1,1953.00,0.00,Fully Paid
7,2025-06-07,4914.00,1,4914.00,0.00,Fully Paid
8,2024-05-13,1044.00,2,1044.00,0.00,Fully Paid
9,2025-12-14,2411.00,1,2411.00,0.00,Fully Paid
10,2024-11-04,3648.00,1,3648.00,0.00,Fully Paid


## Analysis 06 — Revenue Inflation from JOIN Multiplication
### Business question: 
### How much is total order revenue overstated when orders are joined directly to multiple payment transactions without preserving order-level grain?

In [ ]:
%sql
WITH correct_revenue AS (
    SELECT
        SUM(order_amount) AS actual_revenue
    FROM retailops360.source.orders
    WHERE order_status = 'Completed'
),

incorrect_revenue AS (
    SELECT
        SUM(o.order_amount) AS inflated_revenue
    FROM retailops360.source.orders AS o
    INNER JOIN retailops360.source.payments AS p
        ON o.order_id = p.order_id
    WHERE o.order_status = 'Completed'
)

SELECT
    ROUND(c.actual_revenue, 2) AS actual_revenue,
    ROUND(i.inflated_revenue, 2) AS incorrectly_joined_revenue,
    ROUND(
        i.inflated_revenue - c.actual_revenue, 2
    ) AS net_reporting_error,
    ROUND(
        (i.inflated_revenue - c.actual_revenue)
        / NULLIF(c.actual_revenue, 0) * 100,
        2
    ) AS net_error_percentage
FROM correct_revenue AS c
CROSS JOIN incorrect_revenue AS i;

actual_revenue,incorrectly_joined_revenue,net_reporting_error,net_error_percentage
104467399.00,116097952.00,11630553.00,11.13
